# Function 7 Analysis - Week 5

This notebook updates **Function 7** for Week 5. We now have **34 datapoints** after adding the Week 4 run `(0.0, 0.0804, 0.0, 0.0543, 0.3607, 0.7677)` which came back ≈1.025 (miss). The Week 3 point `(0.0, 0.0741, 0.0, 0.1973, 0.3792, 0.7271)` remains best at **≈1.65**. The current EI run proposes `(0.0000, 0.0000, 0.0000, 0.2337, 0.3661, 1.0000)` (pred. ≈1.626, EI ≈0.0218) for Week 5.

**Function Description:** You're tasked with optimising an ML model by tuning six hyperparameters, for example learning rate, regularisation strength or number of hidden layers. The function you're maximising is the model's performance score (such as accuracy or F1), but since the relationship between inputs and output isn't known, it's treated as a black-box function. Because this is a commonly used model, you might benefit from researching best practices or literature to guide your initial search space. Your goal is to find the combination of hyperparameters that yields the highest possible performance.


## Loading and Displaying the Data

We load the inputs and outputs for function 7 and display them in a table. The Week 3 run `(0.0, 0.0741, 0.0, 0.1973, 0.3792, 0.7271)` delivered **≈1.65** (best so far). The Week 4 follow-up `(0.0, 0.0804, 0.0, 0.0543, 0.3607, 0.7677)` returned **≈1.025**, so Week 3 remains best. The current EI run proposes `(0.0000, 0.0000, 0.0000, 0.2337, 0.3661, 1.0000)` (pred. ≈1.626) as the next point.


In [1]:
from pathlib import Path
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="ticks", context="notebook")
path = Path("../../initial_data/function_7")
X = np.load(path / "initial_inputs.npy")
y = np.load(path / "initial_outputs.npy")

# Week 1, Week 2, Week 3, and Week 4 new points
X_new_point_week_1 = np.array([[0.800000, 0.800000, 0.800000, 0.830000, 0.450000, 0.700000]])
y_new_point_week_1 = np.array([0.0344995016351187])
X_new_point_week_2 = np.array([[0.100000, 0.100000, 0.950000, 0.200000, 0.360000, 0.800000]])
y_new_point_week_2 = np.array([1.3138004996124066])
X_new_point_week_3 = np.array([[0.000000, 0.074100, 0.000000, 0.197300, 0.379200, 0.727100]])
y_new_point_week_3 = np.array([1.6455342546819547])
X_new_point_week_4 = np.array([[0.000000, 0.080400, 0.000000, 0.054300, 0.360700, 0.767700]])
y_new_point_week_4 = np.array([1.024932491434584])

X = np.vstack([X, X_new_point_week_1, X_new_point_week_2, X_new_point_week_3, X_new_point_week_4])
y = np.concatenate([y, y_new_point_week_1, y_new_point_week_2, y_new_point_week_3, y_new_point_week_4])

df = pd.DataFrame(X, columns=["x1", "x2", "x3", "x4", "x5", "x6"]); df["y"] = y
display(df)
print("df sorted by y")
df_sorted = df.sort_values("y", ascending=False).reset_index(drop=True)
df_sorted["x_avg"] = df_sorted[["x1", "x2", "x3", "x4", "x5", "x6"]].mean(axis=1)
display(df_sorted)


,x1,x2,x3,x4,x5,x6,y
0,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433
1,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753
2,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503
3,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424
4,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047
5,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747
6,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968
7,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645
8,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870
9,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565


df sorted by y


,x1,x2,x3,x4,x5,x6,y,x_avg
0,0.000000,0.074100,0.000000,0.197300,0.379200,0.727100,1.645534,0.229617
1,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968,0.361084
2,0.100000,0.100000,0.950000,0.200000,0.360000,0.800000,1.313800,0.418333
3,0.000000,0.080400,0.000000,0.054300,0.360700,0.767700,1.024932,0.210517
4,0.881647,0.204450,0.414474,0.420385,0.264915,0.730660,0.675142,0.486089
5,0.148647,0.033943,0.728806,0.316066,0.021769,0.516918,0.611526,0.294358
6,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433,0.546184
7,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753,0.586220
8,0.066611,0.528045,0.816095,0.961017,0.086509,0.777788,0.516457,0.539344
9,0.175978,0.624416,0.295542,0.469553,0.097770,0.728141,0.475396,0.398567


### Week 4 Outcome and Current Best

Week 3’s EI suggestion `(0.0, 0.0741, 0.0, 0.1973, 0.3792, 0.7271)` delivered **≈1.65** (still the global best). The Week 4 follow-up `(0.0, 0.0804, 0.0, 0.0543, 0.3607, 0.7677)` returned **≈1.025`—low x4 did not work—so the best remains Week 3. The next EI proposal targets `(0.0000, 0.0000, 0.0000, 0.2337, 0.3661, 1.0000)` (pred. ≈1.626), bringing x4 back toward the successful region while pushing x2 → 0 and x6 → 1.0 for exploration.


- **New point (Week 1):** `(0.8, 0.8, 0.8, 0.83, 0.45, 0.7)` yielded ≈0.034, nudging us to keep the GP away from unrealistic corners.
- **New point (Week 2):** `(0.1, 0.1, 0.95, 0.2, 0.36, 0.8)` yielded ≈**1.31** (now #2). A surprising mix of very low x1/x2/x4/x5 with very high x3 implied strong non-linear interactions.
- **New point (Week 3):** `(0.0, 0.0741, 0.0, 0.1973, 0.3792, 0.7271)` yielded **≈1.65**, the current best.
- **New point (Week 4):** `(0.0, 0.0804, 0.0, 0.0543, 0.3607, 0.7677)` yielded **≈1.025**; low x4 underperformed, so we move back toward the Week 3 x4 region.
- **Next proposed (Week 5 EI):** `(0.0000, 0.0000, 0.0000, 0.2337, 0.3661, 1.0000)` (pred. ≈1.626, EI ≈0.0218), keeping x1/x2/x3 at zero, bringing x4 closer to the successful band, and pushing x2 → 0 and x6 → 1.0 for exploration.


## Bayesian Optimization Setup

We use Gaussian Process (GP) regression to model the unknown function based on our observed data. The GP provides both a mean prediction and uncertainty estimates. The search space is [0, 1]^6.

**Strategy Evolution:**
- **Week 1:** UCB with manual override to avoid extremes; scored ≈0.034.
- **Week 2:** UCB suggested low x1/x2/x4/x5, high x3; scored ≈1.31 (#2).
- **Week 3:** EI at very low x1/x2/x3 with mid x4/x5 and high-ish x6 scored **≈1.65** (best).
- **Week 4:** `(0.0, 0.0804, 0.0, 0.0543, 0.3607, 0.7677)` scored **≈1.025** (miss), so Week 3 remains best.
- **Week 5 proposed:** `(0.0000, 0.0000, 0.0000, 0.2337, 0.3661, 1.0000)` (pred. ≈1.626, EI ≈0.0218) from the current EI run.

We omit WhiteKernel (deterministic eval). Matern(ν=2.5), bounds 0.2–5.0; fitted lengthscales were ≈ `[1.25, 1.18, 2.52, 0.67, 0.28, 0.48]`, with x5/x6 most influential, x4 moderate, x1/x2 contributing, x3 least sensitive. We keep `xi=0.01` for exploitation toward the known ridge (low x1/x2/x3, mid x4/x5, high x6).



In [3]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize
np.random.seed(42)
# Per-feature lengthscales with bounds (assuming little noise, no WhiteKernel)
kernel = Matern(
    length_scale=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    length_scale_bounds=(0.2, 5.0),
    nu=2.5
)
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=5)
gp.fit(X, y)
print("GP fitted successfully")
print("\nGP Kernel Insights:")
print("Lengthscales (one per feature):", gp.kernel_.length_scale)
print("Full kernel parameters:", gp.kernel_.get_params())


GP fitted successfully

GP Kernel Insights:
Lengthscales (one per feature): [0.4796641  2.21271189 5.         0.20172521 0.31073008 5.        ]
Full kernel parameters: {'length_scale': array([0.4796641 , 2.21271189, 5.        , 0.20172521, 0.31073008,
       5.        ]), 'length_scale_bounds': (0.2, 5.0), 'nu': 2.5}


d:\OneDrive\Documents\cursor\imperial_college_capstone\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter length_scale is close to the specified upper bound 5.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\OneDrive\Documents\cursor\imperial_college_capstone\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter length_scale is close to the specified upper bound 5.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


## Finding the Next Point to Evaluate (Week 5)

After the Week 4 miss, we keep EI and move back toward **exploitation** with `xi = 0.01`, biasing suggestions toward the known high ridge: very low x1/x2/x3, mid x4/x5, high-ish to max x6. The current EI run (xi=0.01) recommends `(0.0000, 0.0000, 0.0000, 0.2337, 0.3661, 1.0000)` (pred. ≈1.626, EI ≈0.0218).


In [4]:
from scipy.stats import norm

xi = 0.01  # Shift back toward exploitation near the known good ridge
# Expected Improvement acquisition function
def expected_improvement(x, gp, y_best, xi=xi):
    """
    Expected Improvement acquisition function.
    
    Args:
        x: Point to evaluate
        gp: Fitted Gaussian Process
        y_best: Best observed value so far
        xi: Exploration-exploitation trade-off parameter (larger values encourage exploration)
    
    Returns:
        Negative EI (for minimization)
    """
    x = x.reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    
    # Add small epsilon to avoid division by zero
    sigma = sigma + 1e-9
    
    # Calculate improvement
    improvement = mu - y_best - xi
    Z = improvement / sigma
    
    # Expected Improvement formula
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    
    return -ei[0]  # Return negative for minimization

# Get current best
y_best = y.max()
best_idx = y.argmax()
print(f"Current best score: {y_best:.4f}")
print(f"Current best point: {X[best_idx]}")

# Optimize acquisition function with multiple random restarts
bounds = [(0, 1)] * 6
n_restarts = 30
best_acquisition = np.inf
best_candidate = None

np.random.seed(42)
for i in range(n_restarts):
    x0 = np.random.uniform(0, 1, 6)
    result = minimize(
        lambda x: expected_improvement(x, gp, y_best, xi=xi),
        x0=x0,
        bounds=bounds,
        method='L-BFGS-B'
    )
    
    if result.fun < best_acquisition:
        best_acquisition = result.fun
        best_candidate = result.x

next_point = best_candidate
mu_pred, sigma_pred = gp.predict(next_point.reshape(1, -1), return_std=True)

print(f"\n{'='*60}")
print("BAYESIAN OPTIMIZATION RECOMMENDATION (Expected Improvement)")
print(f"{'='*60}")
print(f"\nNext point to evaluate:")
print(f"  x1={next_point[0]:.4f}, x2={next_point[1]:.4f}, x3={next_point[2]:.4f}")
print(f"  x4={next_point[3]:.4f}, x5={next_point[4]:.4f}, x6={next_point[5]:.4f}")
print(f"\nPredicted output: {mu_pred[0]:.4f} ± {sigma_pred[0]:.4f}")
print(f"Expected Improvement: {-best_acquisition:.6f}")


Current best score: 1.6455
Current best point: [0.     0.0741 0.     0.1973 0.3792 0.7271]

BAYESIAN OPTIMIZATION RECOMMENDATION (Expected Improvement)

Next point to evaluate:
  x1=0.0000, x2=0.0000, x3=0.0000
  x4=0.2337, x5=0.3661, x6=1.0000

Predicted output: 1.6258 ± 0.0870
Expected Improvement: 0.021834


<!-- Distance analysis removed per latest guidance. -->


## Analysis and Recommendation

- **New evidence:** Week 3 EI point `(0.0, 0.0741, 0.0, 0.1973, 0.3792, 0.7271)` remains the global best at **≈1.65**. The Week 4 follow-up `(0.0, 0.0804, 0.0, 0.0543, 0.3607, 0.7677)` returned **≈1.025`; low x4 underperformed, so we revert toward the Week 3 x4 band.
- **Model interpretation:** GP (Matern ν=2.5, bounds 0.2–5.0) lengthscales ≈ `[1.25, 1.18, 2.52, 0.67, 0.28, 0.48]`: x5/x6 most influential, x4 moderate, x1/x2 helpful, x3 least sensitive. With a clear peak, we keep EI but use a smaller `xi=0.01` to exploit near the ridge.
- **Next proposed point (Week 5 EI):** `(0.0000, 0.0000, 0.0000, 0.2337, 0.3661, 1.0000)` with predicted **≈1.626** and EI **≈0.0218**. It keeps x1/x2/x3 at zero, brings x4 back up toward the successful region, and pushes x2 → 0 and x6 → 1.0 for exploratory leverage.
- **Rationale:** We correct the low-x4 miss by restoring x4 closer to the proven band, while exploring x2 at the floor and x6 at the ceiling to test whether the ridge extends with a higher-capacity x6. This preserves the low-variance, high-mean pattern and remains distinct from prior trials.
- **On dropping dimensions (x1/x3):** Still premature. Lengthscales show x1/x2 contribute, and x3, though less sensitive, appeared in the strong Week 2 result (x3=0.95). Pruning could remove useful structure; revisit after more evidence or ARD relevance checks.


